# Create vector layers

## About this notebook

The source of truth for all vector layer definitions is `vector_config.yaml`. Most layers run straight from config with no manual steps required and are not documented here.

This notebook documents only layers that needed **custom preprocessing** before running the standard processor. These steps are kept for reproducibility and reference.

To process layers that require no preprocessing, use the **working cell** at the bottom of this notebook.

## Setup

### Library import

In [ ]:
import json
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import box

sys.path.append("../src")
from data_processing.process_layers import process_vector_layers

## Preprocessing records

#### Zambezi basin - Water

In [ ]:
# open json file and convert to geojson
with open('../data/raw/Water/Zambezi/Vectors/Luangwa_Streamflow_location.json') as f:
    data = json.load(f)

# save as geojson
data = gpd.GeoDataFrame.from_features(data["features"])
data.set_crs(epsg=4326, inplace=True)
data.to_file(
    '../data/raw/Water/Zambezi/Vectors/Luangwa_Streamflow_location.geojson',
    driver='GeoJSON'
)

#### Bhutan - Health

The gpkg `GDA-Public-Health_RoadN_HFs_Bhutan_2025.gpkg` contains two layers ("Road network" and "Health Faciltiies") that need to be extracted into separate files before processing.

In [ ]:
gpkg_path = "../data/raw/Health/Bhutan/Visuals/Step 2/GDA-Public-Health_RoadN_HFs_Bhutan_2025.gpkg"
output_dir = "../data/raw/Health/Bhutan/Visuals/Step 2"

roads = gpd.read_file(gpkg_path, layer="Road network")
roads.to_file(f"{output_dir}/Road_network.gpkg", driver="GPKG")

facilities = gpd.read_file(gpkg_path, layer="Health Faciltiies")
facilities.to_file(f"{output_dir}/Health_Facilities.gpkg", driver="GPKG")

#### Nigeria - Health

In [ ]:
# Step 2: Merge Kano and Kaduna state boundaries into a single layer
kano = gpd.read_file("../data/raw/Health/Nigeria/Step 2/kano_state.gpkg")
kaduna = gpd.read_file("../data/raw/Health/Nigeria/Step 2/kaduna_state.gpkg")

merged = pd.concat([kano, kaduna], ignore_index=True)
merged = gpd.GeoDataFrame(merged, geometry="geometry", crs=kano.crs)
merged.to_file("../data/raw/Health/Nigeria/Step 2/kano_kaduna_states.gpkg", driver="GPKG")
print(f"Merged {len(merged)} features: {merged['ADM1_EN'].tolist()}")

In [ ]:
# Step 3: Clip crop type map with BBOX (both in EPSG:3857)
bbox_3857 = (932075.8830324067, 1314138.5099005199, 948109.6330324076, 1324086.8432338538)
bbox_geom = box(*bbox_3857)

gdf = gpd.read_file(
    "../data/raw/Health/Nigeria/Step 3/gda-health_nigeria-kano_croptypemap-rice_bu30m.gpkg"
)
clipped = gpd.clip(gdf, bbox_geom)
clipped.to_file(
    "../data/raw/Health/Nigeria/Step 3/croptypemap_rice_clipped.gpkg", driver="GPKG"
)
print(f"Clipped to {len(clipped)} features")

## Process new layers

Use the cell below to process layers that run straight from config (no preprocessing needed).

**Workflow:**
1. Add the layer definition to `vector_config.yaml`
2. Edit `layer_keys` below and run
3. If preprocessing was needed, move those cells to the **Preprocessing records** section above with a markdown header explaining what was done
4. Revert this cell before committing — do not accumulate processed layer keys here

In [ ]:
# WORKING CELL — edit layer_keys, run, then revert. Do not commit with real values.
process_vector_layers(
    config_path="../src/vector_config.yaml",
    layer_keys=[""],
    upload_override=True,
)